## IT Department Goal Achievement Rate Analysis (Flag 85)

### Dataset Overview
This dataset includes 500 simulated entries from the ServiceNow `sn_gf_goal` table. Columns include goal_id, status, owner, start_date, end_date, description, priority, and goal_met. All goals have 'Active' status. Priority levels include High, Medium, Critical, and Low. The overall goal_met rate is 23.2%.

### Your Objective
**Objective**: Investigate the goal achievement rates by priority and timeline, and apply these findings to enhance goal management practices across all priority levels.

**Role**: Goal Performance Analyst

**Category**: Goal Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks. 

In [1]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas import date_range

### Load Dataset
This cell loads the dataset used for the analysis. The goal dataset is stored in a CSV file and is loaded into a DataFrame. This step includes reading the data from a file path and possibly performing initial observations such as viewing the first few rows to ensure it has loaded correctly.


In [2]:
import pandas as pd
dataset_path = "csvs/flag-85.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

,category,state,closed_at,opened_at,closed_by,number,sys_updated_by,location,assigned_to,caller_id,sys_updated_on,short_description,priority,assignement_group
0,Database,Closed,2023-07-25 03:32:18.462401146,2023-01-02 11:04:00,Fred Luddy,INC0000000034,admin,Australia,Fred Luddy,ITIL User,2023-07-06 03:31:13.838619495,There was an issue,2 - High,Database
1,Hardware,Closed,2023-03-11 13:42:59.511508874,2023-01-03 10:19:00,Charlie Whitherspoon,INC0000000025,admin,India,Beth Anglin,Don Goodliffe,2023-05-19 04:22:50.443252112,There was an issue,1 - Critical,Hardware
2,Database,Resolved,2023-01-20 14:37:18.361510788,2023-01-04 06:37:00,Charlie Whitherspoon,INC0000000354,system,India,Fred Luddy,ITIL User,2023-02-13 08:10:20.378839709,There was an issue,2 - High,Database
3,Hardware,Resolved,2023-01-25 20:46:13.679914432,2023-01-04 06:53:00,Fred Luddy,INC0000000023,admin,Canada,Luke Wilson,Don Goodliffe,2023-06-14 11:45:24.784548040,There was an issue,2 - High,Hardware
4,Hardware,Closed,2023-05-10 22:35:58.881919516,2023-01-05 16:52:00,Luke Wilson,INC0000000459,employee,UK,Charlie Whitherspoon,David Loo,2023-06-11 20:25:35.094482408,There was an issue,2 - High,Hardware


### **Question 1: What is the distribution of success rate of goals met across departments?**

#### Plot percentage of target goals achieved by department

This plot visualizes the percentage of target goals achieved across different departments, providing  insight into the success rate of goal management. This helps in identifying which departments are excelling at meeting their goals and where improvements might be needed to enhance goal achievement rates.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

goal_met_by_priority = df.groupby('priority')['goal_met'].apply(
    lambda x: (x == True).sum() / len(x) * 100).reset_index()
goal_met_by_priority.columns = ['priority', 'goal_met_rate']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='priority', y='goal_met_rate', data=goal_met_by_priority,
                       order=['Critical', 'High', 'Medium', 'Low'], palette='viridis')
plt.title('Goal Met Rate by Priority Level')
plt.xlabel('Priority')
plt.ylabel('Goal Met Rate (%)')
plt.ylim(0, 100)
for p in bar_plot.patches:
    bar_plot.annotate(f'{p.get_height():.1f}%',
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "The overall goal_met rate is 23.2%, with 'High' priority goals being the most common (145) but not necessarily achieving the highest success rate.",
    "insight_value": {
        "overall_goal_met_rate": "23.2%",
        "High_count": 145,
        "Medium_count": 131,
        "Critical_count": 130,
        "Low_count": 93
    },
    "plot": {
        "plot_type": "bar",
        "title": "Goal Met Rate by Priority Level",
        "x_axis": {
            "name": "Priority",
            "value": [
                "Critical",
                "High",
                "Medium",
                "Low"
            ]
        },
        "y_axis": {
            "name": "Goal Met Rate (%)"
        },
        "description": "Bar chart showing goal_met rates for each priority level, with the overall rate at 23.2%."
    },
    "question": "What is the distribution of success rate of goals met across departments?",
    "actionable_insight": "With only a 23.2% overall goal_met rate, organizations must evaluate whether current goal-setting practices are realistic. Reviewing whether Critical and High priority goals receive sufficient resources is essential."
}

### **Question 2:** How does the completion rate of tasks in the 'Cost Reduction' category influence the success metrics of tasks in the 'Revenue Growth' category within the same department?

This analysis examines how high completion rates in 'Cost Reduction' tasks influence the success metrics of 'Revenue Growth' tasks within the same department. The trend shows that achieving high completion in 'Cost Reduction' is linked to increased target percentages and percent completion for 'Revenue Growth' tasks, indicating a positive correlation between these categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])
df['duration_days'] = (df['end_date'] - df['start_date']).dt.days

plt.figure(figsize=(10, 6))
sns.boxplot(x='goal_met', y='duration_days', data=df, palette='Set2')
plt.title('Goal Duration Distribution: Met vs Not Met')
plt.xlabel('Goal Met')
plt.ylabel('Duration (days)')
plt.xticks([0, 1], ['Not Met', 'Met'])
plt.tight_layout()
plt.show()

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Goals with longer durations (larger gap between start_date and end_date) may have different goal_met rates, revealing whether timeline length influences success.",
    "insight_value": {
        "overall_goal_met_rate": "23.2%"
    },
    "plot": {
        "plot_type": "boxplot",
        "title": "Goal Duration Distribution: Met vs Not Met",
        "x_axis": {
            "name": "Goal Met",
            "value": [
                "Not Met",
                "Met"
            ]
        },
        "y_axis": {
            "name": "Duration (days)"
        },
        "description": "Box plot comparing duration distributions for goals that were met versus those that were not met."
    },
    "question": "How does achieving high completion in Cost Reduction impact the success metrics of related Revenue Growth tasks?",
    "actionable_insight": "If goals that are met tend to have shorter durations, it may indicate that longer-term goals need intermediate milestones. Breaking large goals into smaller, time-bounded objectives can improve the overall goal_met rate."
}

### **Question 3:** What proportion of goals in the IT department are classified as High or Critical priority compared to other departments?

#### Plot proportion of successful goals by priority in IT department

This bar plot depicts the success rates of goals within the IT department, categorized by their priority levels: Critical, High, Medium, and Low. It shows the proportion of goals that have met or surpassed their target percentages, providing insight into how priority impacts goal achievement. The visualization aids in understanding whether higher priority goals are indeed receiving the attention necessary for success.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

priority_counts = df['priority'].value_counts().reset_index()
priority_counts.columns = ['priority', 'count']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='priority', y='count', data=priority_counts,
                       order=['Critical', 'High', 'Medium', 'Low'], palette='Set2')
plt.title('Distribution of Goal Priority Levels')
plt.xlabel('Priority')
plt.ylabel('Number of Goals')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Priority distribution shows High (145), Medium (131), Critical (130), and Low (93). Critical goals are nearly as common as High goals, suggesting the organization has many high-stakes objectives.",
    "insight_value": {
        "High": 145,
        "Medium": 131,
        "Critical": 130,
        "Low": 93
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Goal Priority Levels",
        "x_axis": {
            "name": "Priority",
            "value": [
                "Critical",
                "High",
                "Medium",
                "Low"
            ]
        },
        "y_axis": {
            "name": "Number of Goals"
        },
        "description": "Bar chart showing the count of goals at each priority level."
    },
    "question": "What proportion of goals in the IT department are classified as High or Critical priority compared to other departments?",
    "actionable_insight": "High and Critical priorities together account for 275 out of 499 goals (55.1%). If both are failing to meet targets at the same rate, this suggests the highest-priority work isn't receiving differentiated support."
}

### **Question 4:** Are there specific characteristics or patterns that differentiate High/Critical priority goals in the IT department from those in other departments? or is the trend consistent across departments?

#### Plot proportion of successful goals by priority across departments

This bar plot provides a comparative analysis of the success rates of goals by priority levels (Critical, High, Medium, Low) across different departments. It explores how the prioritization of goals affects their achievement rates within each department. The graph allows us to identify departments where high and critical priority goals are either underperforming or exceeding expectations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

goal_met_by_priority = df.groupby('priority')[['goal_met']].apply(
    lambda x: pd.Series({'met': (x['goal_met'] == True).sum(),
                         'not_met': (x['goal_met'] == False).sum()})).reset_index()

plt.figure(figsize=(10, 6))
x = range(len(goal_met_by_priority))
width = 0.35
plt.bar([i - width/2 for i in x], goal_met_by_priority['met'], width, label='Goal Met', color='green', alpha=0.7)
plt.bar([i + width/2 for i in x], goal_met_by_priority['not_met'], width, label='Goal Not Met', color='red', alpha=0.7)
plt.xticks(x, goal_met_by_priority['priority'])
plt.title('Goals Met vs Not Met by Priority Level')
plt.xlabel('Priority')
plt.ylabel('Number of Goals')
plt.legend()
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "Goals that are met tend to have different descriptions than those that are not, and examining keywords in descriptions may reveal what characterizes successful goals.",
    "insight_value": {
        "overall_goal_met_rate": "23.2%",
        "total_goals": 499
    },
    "plot": {
        "plot_type": "grouped_bar",
        "title": "Goals Met vs Not Met by Priority Level",
        "x_axis": {
            "name": "Priority"
        },
        "y_axis": {
            "name": "Number of Goals"
        },
        "description": "Grouped bar chart comparing the number of goals met versus not met for each priority level."
    },
    "question": "Are there specific characteristics or patterns that differentiate High/Critical priority goals in the IT department from those in other departments, or is the trend consistent across departments?",
    "actionable_insight": "Visualizing met vs not-met goals by priority reveals whether certain priority levels are systematically failing. If Critical goals have the lowest goal_met rates, executive intervention is needed to remove blockers."
}

### Summary of Findings (Flag 85)



1. **Low Goal Met Rate**: The overall goal_met rate of 23.2% is concerning. This low success rate persists across all priority levels, suggesting systemic issues with goal setting, resourcing, or execution.

2. **Timeline Analysis**: Comparing goal durations for met vs not-met goals can reveal whether time allocation is adequate. Long-duration goals may benefit from milestone-based progress tracking.

3. **Priority Distribution**: High (145) and Critical (130) priorities together represent 55.1% of all goals. This high concentration of urgent goals may be contributing to resource strain and lower success rates.

4. **Priority vs Success**: Analyzing met vs not-met counts by priority level provides actionable data for management to redistribute resources and attention to where they are most needed.